In [ ]:
import marimo as mo

# Welcome to your Strata Notebook

Here you can start authoring your solution to this project.

The required datasets have been preloaded under the `datasets/` directory.
The next two cells demonstrate how you can read the data both in python (with pandas for example) and SQL.

If you want to learn more about how to use Marimo take a look at the [official documentation](https://docs.marimo.io/guides/)

In [ ]:
from pathlib import Path
import pandas as pd

files=Path("datasets").glob("*.csv")
df=pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True
)

In [ ]:
_df = mo.sql(
    f"""
    SELECT * FROM df;
    """
)

In [ ]:
#Aggregating the data on day to day basis

In [ ]:
df['sale_time']=pd.to_datetime(df['sale_time'])
df['sale_day']=df['sale_time'].dt.date
df

In [ ]:
#Finding the total sales for that day

In [ ]:
a=df.groupby('sale_day')['purchaser_gender'].count().reset_index(name="sales")
a

In [ ]:
#Plot daily sales for all 50 weeks.

In [ ]:
import matplotlib.pyplot as plt
a.plot(figsize=(12, 5))
plt.xlabel('Sale Date')
plt.ylabel('Number of Sales')
plt.title('Daily sales over 50 weeks')
plt.show()

In [ ]:
#It looks like there has been a sudden change in daily sales. What date did it occur?
#Will use lag to get generate diff in sales on day to day basis
b=a.copy()
b['sale_day'] = pd.to_datetime(b['sale_day'])

b['previous_sales'] = b['sales'].shift(1)

b['sales_change'] = a['sales'] - b['previous_sales']
max_change=b['sales_change'].max()
sale_change_date=b.loc[b['sales_change'].idxmax(), 'sale_day']
sale_change_date

In [ ]:
#Is the change in daily sales at the date you selected statistically significant? If so, what is the p-value?
from scipy.stats import ttest_ind
b
before=b[b['sale_day']<=sale_change_date]
after=b[b['sale_day']>sale_change_date]
from scipy.stats import ttest_ind

t_stat, p_value = ttest_ind(
    before['sales'],
    after['sales'],
    equal_var=False
)

print(t_stat)
print(p_value)

In [ ]:
# Assume a given day is divided into four dayparts:
import numpy as np

# night (12:00AM - 6:00AM),
# morning (6:00AM - 12:00PM),
# afternoon (12:00PM - 6:00PM),
# evening (6:00PM - 12:00AM).
# What is the percentage of sales in each daypart over all 50 weeks?

df['hour']=df['sale_time'].dt.hour
conditions=[
    (df['hour'] >= 0) & (df['hour'] < 6),
    (df['hour'] >= 6) & (df['hour'] < 12),
    (df['hour'] >= 12) & (df['hour'] < 18),
    (df['hour'] >= 18) & (df['hour'] < 24)
]
df['daypart'] = np.select(
    conditions,
    ['night', 'morning', 'afternoon', 'evening'],
    default='unknown'
)
r = df.groupby('daypart')['sale_time'].count().reset_index(name='purchases')
r['percentage'] = r['purchases'] / r['purchases'].sum() * 100
r


In [ ]:
# Mounts the StrataScratch AI panel: a drawer on the right of the notebook
# with the code tools and, where the environment allows it, project scoring.
# Added by marimo-init rather than kept in the template, so notebooks written
# before the panel existed get it too. Guarded: a notebook must still open on
# an image without the widget installed.
try:
    from strata_ai_widget import AIPanel

    _panel = AIPanel()
except Exception:
    _panel = None
_panel